# AI Model Experiment & Evaluation
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membandingkan dua pendekatan AI dalam menyelesaikan task sentiment analysis ulasan pelanggan:
1. Model klasik Machine Learning (Scikit-learn)
2. LLM API (Gemini)

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

> 🔧 **Catatan Penyesuaian Dataset**
>
> Notebook ini menggunakan dataset `customer_reviews_sentiment.csv` dengan kolom `review_text` dan `sentiment` (nilai: `positif`/`negatif`).
>
> Apabila dataset final berbeda dari yang digunakan saat ini, sesuaikan bagian berikut:
> - Nama file pada `pd.read_csv(...)` di Section 2
> - Nama kolom teks dan label pada Section 3 (saat ini: `review_text`, `sentiment`)
> - Label yang diminta pada prompt LLM di Section 5.2 (saat ini: `positif`/`negatif`)
> - Studi Kasus pada Assignment Brief, apabila domain data berbeda dari e-commerce

## 1. Problem Statement

### Objective

Tujuan dari eksperimen ini adalah membandingkan dua pendekatan AI untuk melakukan sentiment analysis terhadap ulasan pelanggan pada platform e-commerce. Pendekatan yang dibandingkan adalah model Machine Learning klasik menggunakan Scikit-learn dan Large Language Model (LLM) menggunakan Gemini AI API. Perbandingan dilakukan untuk mengetahui performa dan trade-off dari kedua pendekatan sebelum menentukan pendekatan yang sesuai untuk dikembangkan lebih lanjut.

### Target / Label

Target yang diprediksi adalah sentimen dari setiap ulasan pelanggan dengan dua kelas:

- **positif**
- **negatif**

Input model berupa teks ulasan pelanggan pada kolom `review_text`, sedangkan label target terdapat pada kolom `sentiment`.

### Batasan dan Asumsi

Eksperimen menggunakan dataset ulasan pelanggan yang telah disediakan dan kedua pendekatan dievaluasi menggunakan test set yang sama agar perbandingan dapat dilakukan secara adil.

Untuk pendekatan Machine Learning klasik, eksperimen menggunakan preprocessing teks berbasis TF-IDF dan algoritma Logistic Regression.

Untuk pendekatan LLM, klasifikasi dilakukan menggunakan Gemini AI API dengan pendekatan prompting tanpa proses training model.

Evaluasi dilakukan menggunakan Accuracy, Precision, Recall, F1-Score, dan Confusion Matrix.

## 2. Import Library & Load Dataset

In [1]:
# 🔧 Sesuaikan nama file jika dataset final berbeda
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
import os

df = pd.read_csv('../data/customer_reviews_sentiment.csv')
df.head()

,review_id,product_name,review_text,sentiment
0,1,Kemeja Flanel,Pelayanan lambat dan tidak responsif saat diko...,negatif
1,2,Case HP,"Puas banget belanja disini, proses cepat dan b...",positif
2,3,Rice Cooker,"Terima kasih seller, barangnya awet dan sesuai...",positif
3,4,Case HP,Warna produk berbeda jauh dari foto di listing.,negatif
4,5,Headset Bluetooth,"Terima kasih seller, barangnya awet dan sesuai...",positif


In [2]:
# Informasi dasar dataset
print("Jumlah data:", len(df))
print("\nKolom:")
print(df.columns.tolist())

print("\nDistribusi sentiment:")
print(df["sentiment"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

Jumlah data: 200

Kolom:
['review_id', 'product_name', 'review_text', 'sentiment']

Distribusi sentiment:
sentiment
positif    110
negatif     90
Name: count, dtype: int64

Missing values:
review_id       0
product_name    0
review_text     0
sentiment       0
dtype: int64


In [3]:
# Melihat beberapa contoh data
df[["review_text", "sentiment"]].head(10)

,review_text,sentiment
0,Pelayanan lambat dan tidak responsif saat diko...,negatif
1,"Puas banget belanja disini, proses cepat dan b...",positif
2,"Terima kasih seller, barangnya awet dan sesuai...",positif
3,Warna produk berbeda jauh dari foto di listing.,negatif
4,"Terima kasih seller, barangnya awet dan sesuai...",positif
5,"Packing rapi, produk sesuai deskripsi. Recomme...",positif
6,Sudah bayar mahal tapi kualitas jauh dari eksp...,negatif
7,"Belanja disini selalu puas, kualitas terjaga.",positif
8,"Bahan nyaman dipakai, ukuran pas, tidak mengec...",positif
9,"Bahan nyaman dipakai, ukuran pas, tidak mengec...",positif


## 3. Menyiapkan Train/Test Split

Pastikan test set yang sama digunakan untuk kedua pendekatan agar perbandingan adil.

In [ ]:
# 🔧 Sesuaikan nama kolom jika dataset final menggunakan nama kolom berbeda
X = df['review_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', len(X_train))
print('Test size:', len(X_test))

In [9]:
# Train/Test Split

X = df["review_text"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Jumlah data training:", len(X_train))
print("Jumlah data testing:", len(X_test))

Jumlah data training: 160
Jumlah data testing: 40


In [10]:
# TF-IDF Vectorization

vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Ukuran data training:", X_train_tfidf.shape)
print("Ukuran data testing:", X_test_tfidf.shape)

Ukuran data training: (160, 145)
Ukuran data testing: (40, 145)


In [11]:
# Training Logistic Regression

model_lr = LogisticRegression(random_state=42)

model_lr.fit(X_train_tfidf, y_train)

print("Model Logistic Regression berhasil dilatih.")

Model Logistic Regression berhasil dilatih.


In [12]:
# Prediksi pada data test

y_pred_lr = model_lr.predict(X_test_tfidf)

print("Contoh label aktual :", y_test.head(10).tolist())
print("Contoh hasil prediksi:", y_pred_lr[:10].tolist())

Contoh label aktual : ['negatif', 'positif', 'positif', 'positif', 'negatif', 'negatif', 'positif', 'positif', 'negatif', 'negatif']
Contoh hasil prediksi: ['negatif', 'positif', 'positif', 'positif', 'negatif', 'negatif', 'positif', 'positif', 'negatif', 'negatif']


In [13]:
# Evaluasi Logistic Regression

accuracy_lr = accuracy_score(y_test, y_pred_lr)

print("Accuracy :", accuracy_lr)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

Accuracy : 1.0

Classification Report:
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00        18
     positif       1.00      1.00      1.00        22

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



## 4. Pendekatan 1 — Model Klasik (Scikit-learn)

### 4.1 Preprocessing & Feature Extraction

In [14]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

### 4.2 Training Model

In [15]:
clf = LogisticRegression()
clf.fit(X_train_vec, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

### 4.3 Prediksi pada Test Set

In [16]:
y_pred_classic = clf.predict(X_test_vec)

## 5. Pendekatan 2 — LLM API (Gemini)

### Parameter Gemini

Eksperimen menggunakan `temperature=0.2`. Nilai temperature yang relatif rendah dipilih agar output Gemini lebih konsisten dan fokus pada klasifikasi dua kelas sentimen, yaitu **positif** atau **negatif**, serta mengurangi variasi jawaban yang tidak diperlukan.


### 5.1 Setup API

In [41]:
from dotenv import load_dotenv
import os
from google import genai

load_dotenv()

client = genai.Client(
    api_key=os.environ.get("GEMINI_API_KEY")
)

### 5.2 Merancang Prompt

_Tuliskan prompt yang kamu rancang di sini, dan jelaskan alasannya (zero-shot / few-shot)._

In [42]:
def classify_sentiment_llm(review_text):
    prompt = f"""Klasifikasikan sentimen ulasan berikut sebagai "positif" atau "negatif".
Jawab hanya dengan satu kata: positif atau negatif.

Ulasan: "{review_text}"
Sentimen:"""

    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.2)
    )

    return response.text.strip().lower()

In [43]:
# Test fungsi Gemini dengan satu review

test_review = X_test.iloc[0]

result = classify_sentiment_llm(test_review)

print("Review:", test_review)
print("Prediksi Gemini:", result)

Review: Barang tidak berfungsi sama sekali begitu diterima.
Prediksi Gemini: negatif


### 5.3 Menjalankan Prediksi pada Test Set

Catatan: lakukan normalisasi terhadap output LLM sebelum dibandingkan dengan label asli (misal: lowercase, strip whitespace).

In [28]:
# Membagi test set menjadi beberapa batch
batches = [
    X_test.iloc[i:i+10].tolist()
    for i in range(0, len(X_test), 10)
]

print("Jumlah batch:", len(batches))
print("Ukuran setiap batch:", [len(batch) for batch in batches])

Jumlah batch: 4
Ukuran setiap batch: [10, 10, 10, 10]


In [31]:
batch_result = classify_batch_llm(batches[0])

print(batch_result)


1. negatif
2. positif
3. positif
4. positif
5. negatif
6. negatif
7. positif
8. positif
9. negatif
10. negatif


In [30]:
# Prediksi Gemini pada Test Set

def classify_batch_llm(reviews):
    prompt = """Klasifikasikan sentimen setiap ulasan berikut sebagai "positif" atau "negatif".

Aturan:
- Jawab hanya dengan "positif" atau "negatif".
- Berikan satu jawaban untuk setiap ulasan.
- Pertahankan urutan nomor ulasan.
- Format jawaban:
1. positif
2. negatif
3. positif

Ulasan:
"""

    for i, review in enumerate(reviews, start=1):
        prompt += f"\n{i}. {review}"

    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.2
        )
    )

    return response.text.strip()

In [32]:
import re

def parse_batch_predictions(response_text, expected_count):
    predictions = []

    for line in response_text.splitlines():
        match = re.search(r'\b(positif|negatif)\b', line.lower())
        if match:
            predictions.append(match.group(1))

    if len(predictions) != expected_count:
        raise ValueError(
            f"Jumlah prediksi tidak sesuai. "
            f"Expected {expected_count}, tetapi mendapat {len(predictions)}."
        )

    return predictions

In [33]:
batch_predictions = parse_batch_predictions(
    batch_result,
    expected_count=10
)

print("Prediksi batch:", batch_predictions)
print("Jumlah prediksi:", len(batch_predictions))

Prediksi batch: ['negatif', 'positif', 'positif', 'positif', 'negatif', 'negatif', 'positif', 'positif', 'negatif', 'negatif']
Jumlah prediksi: 10


In [34]:
batch_predictions_2 = parse_batch_predictions(
    classify_batch_llm(batches[1]),
    expected_count=10
)

print("Batch 2:", batch_predictions_2)
print("Jumlah:", len(batch_predictions_2))

Batch 2: ['positif', 'negatif', 'positif', 'positif', 'negatif', 'positif', 'negatif', 'positif', 'negatif', 'negatif']
Jumlah: 10


In [35]:
batch_predictions_3 = parse_batch_predictions(
    classify_batch_llm(batches[2]),
    expected_count=10
)

print("Batch 3:", batch_predictions_3)
print("Jumlah:", len(batch_predictions_3))

Batch 3: ['positif', 'negatif', 'positif', 'positif', 'positif', 'positif', 'positif', 'positif', 'positif', 'negatif']
Jumlah: 10


In [36]:
batch_predictions_4 = parse_batch_predictions(
    classify_batch_llm(batches[3]),
    expected_count=10
)

print("Batch 4:", batch_predictions_4)
print("Jumlah:", len(batch_predictions_4))

Batch 4: ['positif', 'negatif', 'positif', 'negatif', 'negatif', 'negatif', 'negatif', 'negatif', 'positif', 'positif']
Jumlah: 10


In [37]:
# Gabungkan seluruh prediksi Gemini

y_pred_llm = (
    batch_predictions
    + batch_predictions_2
    + batch_predictions_3
    + batch_predictions_4
)

print("Jumlah total prediksi Gemini:", len(y_pred_llm))
print("Prediksi Gemini:", y_pred_llm)

Jumlah total prediksi Gemini: 40
Prediksi Gemini: ['negatif', 'positif', 'positif', 'positif', 'negatif', 'negatif', 'positif', 'positif', 'negatif', 'negatif', 'positif', 'negatif', 'positif', 'positif', 'negatif', 'positif', 'negatif', 'positif', 'negatif', 'negatif', 'positif', 'negatif', 'positif', 'positif', 'positif', 'positif', 'positif', 'positif', 'positif', 'negatif', 'positif', 'negatif', 'positif', 'negatif', 'negatif', 'negatif', 'negatif', 'negatif', 'positif', 'positif']


## 6. Evaluasi dan Perbandingan

### 6.1 Evaluasi Model Klasik

In [27]:
print("=== Model Klasik ===")

accuracy_classic = accuracy_score(y_test, y_pred_classic)

print("Accuracy:", accuracy_classic)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_classic))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_classic))

=== Model Klasik ===
Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00        18
     positif       1.00      1.00      1.00        22

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Confusion Matrix:
[[18  0]
 [ 0 22]]


### 6.2 Evaluasi LLM API

In [38]:
# Evaluasi Model Gemini

print("=== Model Gemini ===")

accuracy_llm = accuracy_score(y_test, y_pred_llm)

print("Accuracy:", accuracy_llm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_llm))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_llm))

=== Model Gemini ===
Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00        18
     positif       1.00      1.00      1.00        22

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Confusion Matrix:
[[18  0]
 [ 0 22]]


### 6.3 Tabel Perbandingan Ringkasan

_Susun tabel ringkasan (bisa markdown table atau DataFrame) yang membandingkan Accuracy, Precision, Recall, F1-Score kedua pendekatan._

In [39]:
# Perbandingan Hasil Evaluasi

comparison = pd.DataFrame({
    "Model": ["Scikit-learn (Logistic Regression)", "Gemini AI API"],
    "Accuracy": [accuracy_classic, accuracy_llm],
    "Precision": [
        classification_report(
            y_test, y_pred_classic, output_dict=True
        )["weighted avg"]["precision"],
        classification_report(
            y_test, y_pred_llm, output_dict=True
        )["weighted avg"]["precision"]
    ],
    "Recall": [
        classification_report(
            y_test, y_pred_classic, output_dict=True
        )["weighted avg"]["recall"],
        classification_report(
            y_test, y_pred_llm, output_dict=True
        )["weighted avg"]["recall"]
    ],
    "F1-Score": [
        classification_report(
            y_test, y_pred_classic, output_dict=True
        )["weighted avg"]["f1-score"],
        classification_report(
            y_test, y_pred_llm, output_dict=True
        )["weighted avg"]["f1-score"]
    ]
})

comparison

,Model,Accuracy,Precision,Recall,F1-Score
0,Scikit-learn (Logistic Regression),1.0,1.0,1.0,1.0
1,Gemini AI API,1.0,1.0,1.0,1.0


## 7. Analisis Trade-off dan Limitation


#### Performa

Berdasarkan hasil eksperimen pada 40 data test, model Logistic Regression berbasis TF-IDF dan Gemini AI API menghasilkan performa yang sama. Kedua pendekatan memperoleh Accuracy, Precision, Recall, dan F1-Score sebesar 1.00. Confusion matrix kedua model juga menunjukkan 18 data negatif dan 22 data positif berhasil diklasifikasikan dengan benar.

Hasil ini menunjukkan bahwa pada dataset dan test set yang digunakan dalam eksperimen ini, kedua pendekatan mampu melakukan klasifikasi sentimen dengan hasil yang sama. Namun, hasil tersebut tidak dapat langsung digeneralisasikan ke dataset lain atau data produksi karena eksperimen hanya menggunakan 40 data test.

#### Effort Implementasi

Model klasik membutuhkan beberapa tahap seperti preprocessing teks menggunakan TF-IDF, pembagian dataset, dan proses training Logistic Regression. Setelah model selesai dilatih, prediksi dapat dilakukan secara lokal tanpa perlu mengirim data ke layanan eksternal.

Gemini AI API tidak membutuhkan proses training model. Implementasinya lebih sederhana dari sisi model karena klasifikasi dilakukan melalui prompt dan API. Namun, diperlukan konfigurasi API dan pengelolaan request ke layanan Gemini.

#### Kecepatan

Model Logistic Regression dapat melakukan prediksi secara lokal setelah model selesai dilatih sehingga tidak bergantung pada koneksi internet atau response time dari API.

Gemini membutuhkan request ke API untuk setiap proses inference. Pada eksperimen ini, pemanggilan API juga perlu dilakukan secara bertahap karena adanya batas penggunaan API. Oleh karena itu, penggunaan Gemini perlu mempertimbangkan latency dan rate limit.

#### Biaya

Model klasik tidak membutuhkan biaya API ketika digunakan untuk melakukan prediksi secara lokal setelah model dilatih. Biaya utamanya lebih berkaitan dengan resource komputasi yang digunakan untuk menjalankan model.

Gemini AI API berpotensi menimbulkan biaya penggunaan berdasarkan jumlah request dan penggunaan token, tergantung layanan dan konfigurasi API yang digunakan.

#### Limitation

Keterbatasan model klasik adalah performanya bergantung pada dataset training dan representasi fitur yang digunakan. Model juga perlu dilatih kembali apabila ingin disesuaikan dengan karakteristik data yang berbeda.

Keterbatasan Gemini dalam eksperimen ini adalah ketergantungan terhadap API, koneksi internet, dan rate limit. Selama eksperimen, pemanggilan Gemini sempat mengalami error `429 RESOURCE_EXHAUSTED`, sehingga inference 40 data perlu dilakukan menggunakan beberapa batch.

Selain itu, kedua model memperoleh hasil sempurna pada test set yang hanya terdiri dari 40 data. Oleh karena itu, hasil ini belum cukup untuk menyatakan bahwa kedua pendekatan akan memiliki performa yang sama pada dataset yang lebih besar atau data dunia nyata.

## 8. Rekomendasi Technical Approach

Berdasarkan hasil eksperimen, model Logistic Regression berbasis TF-IDF dan Gemini AI API menghasilkan performa yang sama pada test set yang digunakan. Kedua pendekatan memperoleh Accuracy, Precision, Recall, dan F1-Score sebesar 1.00.

Untuk use case klasifikasi sentimen sederhana pada dataset ini, pendekatan model klasik berbasis TF-IDF dan Logistic Regression direkomendasikan untuk dikembangkan lebih lanjut. Pertimbangan utamanya adalah model dapat dijalankan secara lokal setelah proses training, tidak bergantung pada API eksternal untuk inference, serta tidak menghadapi rate limit API pada saat melakukan prediksi.

Sementara itu, Gemini AI API tetap memiliki keunggulan dari sisi kemudahan implementasi karena tidak membutuhkan proses training model. Pendekatan ini dapat menjadi alternatif ketika kebutuhan sistem membutuhkan kemampuan pemahaman bahasa yang lebih fleksibel atau ketika proses training dan pemeliharaan model lokal ingin diminimalkan.

Rekomendasi ini didasarkan pada hasil eksperimen saat ini. Karena pengujian hanya dilakukan pada 40 data test dan kedua pendekatan memperoleh hasil yang sama, pengujian dengan dataset yang lebih besar dan lebih beragam tetap diperlukan sebelum menentukan pendekatan untuk lingkungan produksi.